# 1. Importing libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_union
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import balanced_accuracy_score, classification_report

import optuna

/Users/alvinwong/anaconda3/envs/py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA = 'data/'

# 2. Model training

## 2.1 Vectorize

### 2.1a TF-IDF vectorizer

In [3]:
tfidf_vectorizer = make_union(
    TfidfVectorizer(analyzer='char', ngram_range=(2, 5), sublinear_tf=True,
                    min_df=2, max_features=300_000, lowercase=False),
    TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True,
                    min_df=2, lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]"),
)

### 2.1b Count vectorizer

In [4]:
count_vectorizer = make_union(
    CountVectorizer(analyzer='char', ngram_range=(2, 5),
                    min_df=2, max_features=300_000, lowercase=False),
    CountVectorizer(analyzer='word', ngram_range=(1, 2),
                    min_df=2, lowercase=False, token_pattern=r"(?u)\b\w+\b|[^\w\s]"),
)

## 2.2 Classifier models

In [5]:
def train_model(model, vectorizer, X_train, y_train, X_val, y_val):
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)

    model.fit(X_train_vec, y_train)
    
    y_pred = model.predict(X_val_vec)

    return y_pred

### 2.2a SVM

In [6]:
svm = LinearSVC(C=0.5, class_weight='balanced')

### 2.2b Logistic regression  

In [7]:
log_reg = LogisticRegression(C=10, class_weight='balanced', max_iter=2000)
# log_reg = LogisticRegression(C=1, max_iter=300, class_weight='balanced', verbose=1)

## 2.3 Training the model

In [8]:
def training_pipeline(df, vectorizer, classifier, vec_name, classifier_name, extra_data = None):
    all_preds = []

    for fold, (train_idx, val_idx) in enumerate(GroupKFold(n_splits=5).split(df, groups=df['prompt_id'])):
        train_df = df.iloc[train_idx].copy()
        val_df = df.iloc[val_idx].copy()

        X_train, y_train = train_df['text'], train_df['model']
        X_val, y_val = val_df['text'], val_df['model']

        if extra_data is not None:                                  # add external data to training only
            X_train = pd.concat([X_train, extra_data['text']], ignore_index=True)
            y_train = pd.concat([y_train, extra_data['model']], ignore_index=True)

        prediction = train_model(
            clone(classifier),
            clone(vectorizer),
            X_train, y_train, X_val, y_val
        )

        val_df['pred'] = prediction
        val_df['vectorizer'] = vec_name
        val_df['classifier'] = classifier_name
        val_df['fold'] = fold
        all_preds.append(val_df)

    return pd.concat(all_preds)

In [9]:
df = pd.read_csv(f'{DATA}/train.csv')

vectorizers = {
    'TF-IDF': tfidf_vectorizer,
    'Count': count_vectorizer,
}

classifiers = {
    'SVM': svm, 
    'Logistic Regression': log_reg,
}

all_predictions = []
for vec_name, vectorizer in vectorizers.items():
    for classifier_name, classifier in classifiers.items():
        preds = training_pipeline(df, vectorizer, classifier, vec_name, classifier_name)
        all_predictions.append(preds)

predictions_df = pd.concat(all_predictions)

In [10]:
overall_df = pd.DataFrame(
    predictions_df.groupby(['vectorizer', 'classifier'])
                  .apply(lambda g: balanced_accuracy_score(g['model'], g['pred']))
                  .round(3),
    columns=['balanced_accuracy']
)
overall_df = overall_df.sort_values(by="balanced_accuracy", ascending=False).reset_index()
overall_df

/var/folders/_d/vbh48gmd31s0b7r48gvh7m3w0000gn/T/ipykernel_49324/3722617318.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: balanced_accuracy_score(g['model'], g['pred']))


,vectorizer,classifier,balanced_accuracy
0,TF-IDF,Logistic Regression,0.346
1,TF-IDF,SVM,0.338
2,Count,Logistic Regression,0.307
3,Count,SVM,0.293


In [11]:
predictions_df['correct'] = predictions_df['model'] == predictions_df['pred']

recall_df = predictions_df.pivot_table(
    index='model',
    columns=['vectorizer', 'classifier'],
    values='correct',
    aggfunc='mean'
)

recall_df.columns = [f'{v} + {c}' for v, c in recall_df.columns]

recall_df = (recall_df
             .sort_values('TF-IDF + Logistic Regression', ascending=False)
             .round(2)
             .reset_index())
recall_df

,model,Count + Logistic Regression,Count + SVM,TF-IDF + Logistic Regression,TF-IDF + SVM
0,gemini-3.1-pro-preview,0.67,0.64,0.65,0.74
1,claude-opus-5,0.45,0.39,0.65,0.67
2,gpt-6-astra,0.49,0.50,0.49,0.54
3,gemini-3.6-flash,0.37,0.35,0.45,0.43
4,gemini-3.7-flash,0.39,0.36,0.45,0.43
5,claude-opus-4-6,0.37,0.33,0.41,0.42
6,gpt-5.6-luna,0.22,0.17,0.37,0.28
7,qwen3.8-max,0.31,0.31,0.37,0.35
8,claude-sonnet-5,0.27,0.30,0.36,0.35
9,gemini-3.8-flash,0.32,0.28,0.33,0.31


# 3. Optuna tuning 

In [12]:
def objective(trial):
    C = trial.suggest_float('C', 0.01, 70, log=True)

    model = LogisticRegression(
        C=C, 
        class_weight='balanced', 
        max_iter=2000
    )

    preds = training_pipeline(df, tfidf_vectorizer, model, 'TF-IDF', 'Logistic Regression')
    return balanced_accuracy_score(preds['model'], preds['pred'])

study = optuna.create_study(direction='maximize')
study.enqueue_trial({'C': 10})
study.optimize(objective, n_trials=30)

print(study.best_params, round(study.best_value, 3))

[I 2026-11-06 17:48:16,086] A new study created in memory with name: no-name-0ad4e91d-d2e3-4846-b3dc-3a456e4251e6
[I 2026-11-06 17:49:20,378] Trial 0 finished with value: 0.3459681829480487 and parameters: {'C': 10}. Best is trial 0 with value: 0.3459681829480487.
[I 2026-11-06 17:49:36,750] Trial 1 finished with value: 0.2940939597315436 and parameters: {'C': 0.01841554852846756}. Best is trial 0 with value: 0.3459681829480487.
[I 2026-11-06 17:50:38,374] Trial 2 finished with value: 0.34337310464827236 and parameters: {'C': 3.4115463672587487}. Best is trial 0 with value: 0.3459681829480487.
[I 2026-11-06 17:50:56,021] Trial 3 finished with value: 0.2959458115833955 and parameters: {'C': 0.04199283653581233}. Best is trial 0 with value: 0.3459681829480487.
[I 2026-11-06 17:51:18,864] Trial 4 finished with value: 0.31039771314939096 and parameters: {'C': 0.2592199251694285}. Best is trial 0 with value: 0.3459681829480487.
[I 2026-11-06 17:52:19,485] Trial 5 finished with value: 0.3437

{'C': 31.52375801406866} 0.349


In [13]:
best_C = study.best_params['C']

best_model = LogisticRegression(
    C=best_C,
    class_weight='balanced',
    max_iter=2000
)

# 4. Adding new data

In [14]:
EXTRA = f'{DATA}/nlm-matched-clean/'

extra_data = pd.read_parquet(f'{EXTRA}/nlm_pool_v5.parquet')   
extra_data

,text,model,prompt_key,src,w_host
0,Many people believe that a car's rubber tyres ...,claude-fable-5,seyit:g0000,seyitkaangunes/ntl-extra-api-passages,1.0
1,Many people believe that during a lightning st...,claude-opus-4-6,seyit:g0000,seyitkaangunes/ntl-extra-api-passages,1.0
2,It's a common belief that the rubber tyres on ...,claude-opus-4-8,seyit:g0000,seyitkaangunes/ntl-extra-api-passages,1.0
3,You're safe in a car because its metal shell a...,claude-opus-5,seyit:g0000,seyitkaangunes/ntl-extra-api-passages,1.0
4,"When lightning strikes a car, you're protected...",claude-sonnet-5,seyit:g0000,seyitkaangunes/ntl-extra-api-passages,1.0
...,...,...,...,...,...
119894,The papyrus trembled beneath his fingers as th...,claude-sonnet-5,,own_gen,2.0
119895,A neutron star is what remains when a giant su...,gpt-5.6-terra,,own_gen,2.0
119896,A white hole is a hypothetical spacetime regio...,gpt-5.6-sol,,own_gen,2.0
119897,Plasma is matter turned so energetic that atom...,gpt-5.6-terra,,own_gen,2.0


In [16]:
print(extra_data["model"].nunique())
print(extra_data["prompt_key"].nunique())

18
14964


In [15]:
df = pd.read_csv(f'{DATA}/train.csv')

preds = training_pipeline(
    df, tfidf_vectorizer, best_model, 
    'TF-IDF', 'Logistic Regression', 
    extra_data=extra_data
)

score = balanced_accuracy_score(preds['model'], preds['pred'])
print(f'Balanced accuracy = {score:.3f}')

# recall per model
preds['correct'] = preds['model'] == preds['pred']

recall_df = (preds.groupby('model')['correct']
                  .mean()
                  .round(2)
                  .sort_values(ascending=False)
                  .reset_index(name='recall'))
recall_df

Balanced accuracy = 0.462


,model,recall
0,gemini-3.1-pro-preview,0.71
1,gemini-3.6-flash,0.68
2,gemini-3.8-flash,0.65
3,claude-opus-4-6,0.65
4,claude-sonnet-5,0.60
5,gpt-6-astra,0.57
6,gpt-5.6-luna,0.55
7,qwen3.8-max,0.54
8,claude-opus-5,0.52
9,gpt-5.6-terra,0.42
